* list down all the folders

In [1]:
from pathlib import Path

base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")

folders = [p.name for p in base_path.iterdir() if p.is_dir()]

for folder in folders:
    print(folder)


sub-01
sub-02
sub-03
sub-04
sub-05
sub-06
sub-07
sub-08
sub-09
sub-10
sub-11
sub-12
sub-13
sub-14
sub-15
sub-16
sub-17
sub-18
sub-19
sub-20
sub-21
sub-22
sub-23
sub-24
sub-25
sub-26
sub-27
sub-28
sub-29
sub-30
sub-31
sub-32
sub-33
sub-34
sub-35
sub-36
sub-37
sub-38
sub-39
sub-40
sub-41
sub-42
sub-43
sub-44
sub-45
sub-46
sub-47
sub-48
sub-49
sub-50
sub-51
sub-53
sub-54
sub-56
sub-57
sub-58
sub-59
sub-60
sub-61
sub-62
sub-63
sub-64
sub-65
sub-66
sub-67
sub-68
sub-70
sub-71
sub-72
sub-73
sub-74
sub-75
sub-76
sub-77
sub-78
sub-79
sub-80


* load sub-01 EEG data

In [2]:
pip install mne

Note: you may need to restart the kernel to use updated packages.


In [3]:
import mne
from pathlib import Path

# Path to subject folder
sub_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data\sub-01")

# Find the .vhdr file
vhdr_file = list(sub_path.glob("*.vhdr"))[0]

# Load EEG data
raw = mne.io.read_raw_brainvision(vhdr_file, preload=True)

# Print basic info
print(raw)


Extracting parameters from D:\Dermerzel\SomnasNest\Alzheimer\Data\sub-01\sub-01_task-rest_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 661519  =      0.000 ...   661.519 secs...
<RawBrainVision | sub-01_task-rest_eeg.eeg, 127 x 661520 (661.5 s), ~641.1 MiB, data loaded>


In [4]:
# Show channel names
print(raw.ch_names)

['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2', 'F9', 'AFF1h', 'FFC1h', 'FFC5h', 'FTT7h', 'FCC3h', 'CCP1h', 'CCP5h', 'TPP7h', 'P9', 'PPO9h', 'PO9', 'O9', 'OI1h', 'PPO1h', 'CPP3h', 'CPP4h', 'PPO2h', 'OI2h', 'O10', 'PO10', 'PPO10h', 'P10', 'TPP8h', 'CCP6h', 'CCP2h', 'FCC4h', 'FTT8h', 'FFC6h', 'FFC2h', 'AFF2h', 'F10', 'AFp1', 'AFF5h', 'FFT9h', 'FFT7h', 'FFC3h', 'FCC1h', 'FCC5h', 'FTT9h', 'TTP7h', 'CCP3h', 'CPP1h', 'CPP5h', 'TPP9h', 'POO9h', 'PPO5h', 'POO1', 'POO2', 'PPO6h', 'POO10h', 'TPP10h', 'CPP6h', 'CPP2h', 'CCP4h', 'TTP8h', 'FTT10h', 'FCC6h', 'FCC2h', 'FFC4h', 'FFT8h', 'FFT10h', 'AFF6h', 'AFp2']


* save ngeative class subjects as numpy array

In [5]:
import mne
import numpy as np
from pathlib import Path

print("=== EEG Loading Started ===")

# Base directory
base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")
print(f"Base path set to: {base_path}")

TARGET_LENGTH = 150_000  # <-- FIXED NUMBER OF TIME POINTS

# 🔹 ELECTRODES TO KEEP
SELECTED_CHANNELS = [
    "CP1", "CP2", "CP5", "CP6",
    "T7", "T8",
    "P3", "P4", "Pz",
    "PO3", "PO4", "PO7", "PO8",
    "O1", "O2", "Oz"
]

eeg_data_list = []
loaded_subjects = []

# --------------------------------------------------
# STEP 1: Load and crop each subject
# --------------------------------------------------
for i in range(1, 32):
    sub_id = f"sub-{i:02d}"
    sub_path = base_path / sub_id

    print("\n----------------------------------")
    print(f"Processing {sub_id}")
    print(f"Looking in: {sub_path}")

    if not sub_path.exists():
        print(f"❌ Folder not found: {sub_path}")
        continue

    vhdr_files = list(sub_path.glob("*.vhdr"))
    print(f"Found {len(vhdr_files)} .vhdr file(s)")

    if len(vhdr_files) == 0:
        print(f"⚠️ No .vhdr file found in {sub_id}, skipping.")
        continue

    vhdr_file = vhdr_files[0]
    print(f"Using file: {vhdr_file.name}")

    try:
        print("→ Loading EEG data...")
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=True, verbose=False)

        print("→ Picking selected EEG channels only...")
        raw.pick_channels(SELECTED_CHANNELS)

        print("→ Extracting NumPy array...")
        data = raw.get_data()

        n_channels, n_times = data.shape
        print(f"✔ Original shape after channel selection: {data.shape}")

        if n_times < TARGET_LENGTH:
            print(f"⚠️ Skipping {sub_id}: only {n_times} samples (< {TARGET_LENGTH})")
            continue

        # KEEP FIRST 150,000 SAMPLES ONLY
        cropped = data[:, :TARGET_LENGTH].astype(np.float16)

        print(f"✔ Cropped shape: {cropped.shape}")

        eeg_data_list.append(cropped)
        loaded_subjects.append(sub_id)

        print(f"✔ {sub_id} successfully processed.")

    except Exception as e:
        print(f"❌ Error loading {sub_id}: {e}")
        continue

print("\n==================================")
print("EEG Loading Finished")
print(f"Total subjects loaded: {len(eeg_data_list)}")
print(f"Subjects: {loaded_subjects}")

# --------------------------------------------------
# STEP 2: Stack all subjects
# --------------------------------------------------
print("\n→ Stacking all subjects into one NumPy array...")

try:
    ad_negative = np.stack(eeg_data_list, axis=0)
    print("✔ Stacking successful.")
    print("Final shape of ad_negative:", ad_negative.shape)
    print("Data type:", ad_negative.dtype)
except MemoryError as e:
    print("❌ Memory error during stacking!")
    print(e)

=== EEG Loading Started ===
Base path set to: D:\Dermerzel\SomnasNest\Alzheimer\Data

----------------------------------
Processing sub-01
Looking in: D:\Dermerzel\SomnasNest\Alzheimer\Data\sub-01
Found 1 .vhdr file(s)
Using file: sub-01_task-rest_eeg.vhdr
→ Loading EEG data...
ERROR! Session/line number was not unique in database. History logging moved to new session 469
→ Picking selected EEG channels only...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
→ Extracting NumPy array...
✔ Original shape after channel selection: (16, 661520)
✔ Cropped shape: (16, 150000)
✔ sub-01 successfully processed.

----------------------------------
Processing sub-02
Looking in: D:\Dermerzel\SomnasNest\Alzheimer\Data\sub-02
Found 1 .vhdr file(s)
Using file: sub-02_task-rest_eeg.vhdr
→ Loading EEG data...
→ Picking selected EEG channels only...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
→ Extracting NumPy array...
✔ Original shape 

In [7]:
import numpy as np

save_path = r"D:\Dermerzel\SomnasNest\Alzheimer\Data-v2\ad_negative.npy"

np.save(save_path, ad_negative)

print(f"✔ ad_negative saved successfully to:\n{save_path}")
print("Saved shape:", ad_negative.shape)
print("Saved dtype:", ad_negative.dtype)


✔ ad_negative saved successfully to:
D:\Dermerzel\SomnasNest\Alzheimer\Data-v2\ad_negative.npy
Saved shape: (31, 16, 150000)
Saved dtype: float16


* saave the positive class as numpy array

In [8]:
import mne
import numpy as np
from pathlib import Path

print("=== EEG Loading Started (AD POSITIVE) ===")

# Base directory (where subject folders are)
base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")
print(f"Base path set to: {base_path}")

# Save directory (NEW LOCATION)
save_base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data-v2")
save_base_path.mkdir(parents=True, exist_ok=True)

TARGET_LENGTH = 150_000

# 🔹 SELECTED ELECTRODES ONLY
SELECTED_CHANNELS = [
    "CP1", "CP2", "CP5", "CP6",
    "T7", "T8",
    "P3", "P4", "Pz",
    "PO3", "PO4", "PO7", "PO8",
    "O1", "O2", "Oz"
]

eeg_data_list = []
loaded_subjects = []

# --------------------------------------------------
# STEP 1: Load and crop each subject
# --------------------------------------------------
for i in range(32, 81):
    sub_id = f"sub-{i:02d}"
    sub_path = base_path / sub_id

    print("\n----------------------------------")
    print(f"Processing {sub_id}")
    print(f"Looking in: {sub_path}")

    if not sub_path.exists():
        print(f"❌ Folder not found: {sub_path}")
        continue

    vhdr_files = list(sub_path.glob("*.vhdr"))
    print(f"Found {len(vhdr_files)} .vhdr file(s)")

    if len(vhdr_files) == 0:
        print(f"⚠️ No .vhdr file found in {sub_id}, skipping.")
        continue

    vhdr_file = vhdr_files[0]
    print(f"Using file: {vhdr_file.name}")

    try:
        print("→ Loading EEG data...")
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=True, verbose=False)

        print("→ Picking selected EEG channels only...")
        raw.pick_channels(SELECTED_CHANNELS)

        print("→ Extracting NumPy array...")
        data = raw.get_data()

        n_channels, n_times = data.shape
        print(f"✔ Shape after channel selection: {data.shape}")

        if n_times < TARGET_LENGTH:
            print(f"⚠️ Skipping {sub_id}: only {n_times} samples (< {TARGET_LENGTH})")
            continue

        # KEEP FIRST 150,000 SAMPLES
        cropped = data[:, :TARGET_LENGTH].astype(np.float16)

        print(f"✔ Cropped shape: {cropped.shape}")

        eeg_data_list.append(cropped)
        loaded_subjects.append(sub_id)

        print(f"✔ {sub_id} successfully processed.")

    except Exception as e:
        print(f"❌ Error loading {sub_id}: {e}")
        continue

print("\n==================================")
print("EEG Loading Finished (AD POSITIVE)")
print(f"Total subjects loaded: {len(eeg_data_list)}")
print(f"Subjects: {loaded_subjects}")

# --------------------------------------------------
# STEP 2: Stack all subjects
# --------------------------------------------------
print("\n→ Stacking all subjects into one NumPy array...")

try:
    ad_positive = np.stack(eeg_data_list, axis=0)
    print("✔ Stacking successful.")
    print("Final shape of ad_positive:", ad_positive.shape)
    print("Data type:", ad_positive.dtype)
except MemoryError as e:
    print("❌ Memory error during stacking!")
    print(e)
    raise

# --------------------------------------------------
# STEP 3: Save to disk (NEW LOCATION)
# --------------------------------------------------
save_path = save_base_path / "ad_positive.npy"

np.save(save_path, ad_positive)

print(f"\n✔ ad_positive saved successfully to:")
print(save_path)

=== EEG Loading Started (AD POSITIVE) ===
Base path set to: D:\Dermerzel\SomnasNest\Alzheimer\Data

----------------------------------
Processing sub-32
Looking in: D:\Dermerzel\SomnasNest\Alzheimer\Data\sub-32
Found 1 .vhdr file(s)
Using file: sub-32_task-rest_eeg.vhdr
→ Loading EEG data...
→ Picking selected EEG channels only...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
→ Extracting NumPy array...
✔ Shape after channel selection: (16, 625680)
✔ Cropped shape: (16, 150000)
✔ sub-32 successfully processed.

----------------------------------
Processing sub-33
Looking in: D:\Dermerzel\SomnasNest\Alzheimer\Data\sub-33
Found 1 .vhdr file(s)
Using file: sub-33_task-rest_eeg.vhdr
→ Loading EEG data...
→ Picking selected EEG channels only...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
→ Extracting NumPy array...
✔ Shape after channel selection: (16, 658300)
✔ Cropped shape: (16, 150000)
✔ sub-33 successfully processed.

In [9]:
import mne
from pathlib import Path

print("=== Checking EEG sampling rates from sub-01 to sub-80 ===")

base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")

sampling_rates = {}

for i in range(1, 81):
    sub_id = f"sub-{i:02d}"
    sub_path = base_path / sub_id

    print("\n----------------------------------")
    print(f"Processing {sub_id}")

    if not sub_path.exists():
        print("❌ Folder not found")
        continue

    vhdr_files = list(sub_path.glob("*.vhdr"))
    if len(vhdr_files) == 0:
        print("⚠️ No .vhdr file found")
        continue

    vhdr_file = vhdr_files[0]
    print(f"Using file: {vhdr_file.name}")

    try:
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=False, verbose=False)
        raw.pick_types(eeg=True)

        sfreq = raw.info["sfreq"]
        sampling_rates[sub_id] = sfreq

        print(f"✔ Sampling rate: {sfreq} Hz")

    except Exception as e:
        print(f"❌ Error loading {sub_id}: {e}")

print("\n==================================")
print("=== Sampling Rate Summary ===")

if sampling_rates:
    unique_sfreqs = sorted(set(sampling_rates.values()))

    print(f"Subjects successfully loaded: {len(sampling_rates)}")
    print(f"Unique sampling rates (Hz): {unique_sfreqs}")

    print("\nPer-subject sampling rates:")
    for sub, sfreq in sampling_rates.items():
        print(f"  {sub} → {sfreq} Hz")
else:
    print("❌ No valid EEG data found")


=== Checking EEG sampling rates from sub-01 to sub-80 ===

----------------------------------
Processing sub-01
Using file: sub-01_task-rest_eeg.vhdr
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
✔ Sampling rate: 1000.0 Hz

----------------------------------
Processing sub-02
Using file: sub-02_task-rest_eeg.vhdr
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
✔ Sampling rate: 1000.0 Hz

----------------------------------
Processing sub-03
Using file: sub-03_task-rest_eeg.vhdr
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
✔ Sampling rate: 1000.0 Hz

----------------------------------
Processing sub-04
Using file: sub-04_task-rest_eeg.vhdr
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
✔ Sampling rate: 1000.0 Hz

----------------------------------
Processing sub-05
Using file: sub-05_task-rest_eeg.vhdr
NOTE: pick_types() is a legacy function. New code should use inst.